In [290]:
!pip install chromadb langchain-text-splitters -qq

In [291]:
import os
import time
from typing import List, Dict, Any
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
import json

In [292]:
# Ground truth policies (one large file)
policy_documents = [
        {
            "id": "policy_001",
            "title": "Home Office Equipment Reimbursement",
            "content": "Employees working from home may claim up to $500 per year for office equipment including desks, chairs, monitors, and computer accessories. Receipts must be submitted within 30 days of purchase. This policy applies to full-time remote workers only. The equipment must be used primarily for work purposes and should be ergonomic and suitable for a professional home office environment.",
            "category": "reimbursement"
        },
        {
            "id": "policy_002",
            "title": "Travel Expense Guidelines",
            "content": "Business travel expenses are reimbursable when pre-approved by your manager. Meals are covered up to $50 per day, hotel stays up to $200 per night. All receipts must be submitted within 14 days of return. International travel requires additional approval from the department head. Travel insurance is mandatory for all business trips exceeding 7 days.",
            "category": "travel"
        },
        {
            "id": "policy_003",
            "title": "Remote Work Furniture Policy",
            "content": "Remote employees may purchase ergonomic furniture for their home office setup. This includes standing desks, ergonomic chairs, and monitor arms. Maximum reimbursement is $300 per item with manager approval required. All furniture must meet ergonomic standards and be purchased from approved vendors. Receipts must be submitted within 45 days of purchase.",
            "category": "reimbursement"
        },
        {
            "id": "policy_004",
            "title": "Equipment and Supplies Reimbursement",
            "content": "Work-related equipment and supplies purchased for home office use are eligible for reimbursement. This covers laptops, monitors, keyboards, mice, and other computer peripherals. Submit expense reports with receipts for approval. Equipment must be used for work purposes and should be compatible with company systems. Annual limit is $1000 per employee.",
            "category": "reimbursement"
        },
        {
            "id": "policy_005",
            "title": "Vacation and PTO Policy",
            "content": "Full-time employees accrue 15 days of paid time off per year. Vacation requests must be submitted at least 2 weeks in advance. Unused PTO does not roll over to the next year. Emergency leave can be taken with manager approval. Sick leave is separate from vacation time and does not count against PTO balance.",
            "category": "benefits"
        }
    ]

In [293]:
def load_and_chunk_documents():
    global policy_documents
    print("📚 SECTION 1: DOCUMENT LOADING & CHUNKING")
    print("-" * 50)
    print(f"📄 Loaded {len(policy_documents)} policy documents")

    # Create text splitter object
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=50,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    all_chunks = []
    for doc in policy_documents:
      chunks = text_splitter.split_text(doc["content"])
      for i, chunk in enumerate(chunks):
            all_chunks.append({
                "id": f"{doc['id']}_chunk_{i}",
                "title": doc["title"],
                "content": chunk,
                "category": doc["category"],
                "source_doc": doc["id"]
            })
    print(f"Splited to {len(all_chunks)} chunks")
    return all_chunks


In [294]:
def setup_vector_database(chunks: List[Dict]):
    print("\n🗄️ SECTION 2: VECTOR DATABASE SETUP")
    print("-" * 50)

    # Initialize ChromaDB client
    client = chromadb.PersistentClient(path="./my_chroma_db")

    model_name = "all-MiniLM-L6-v2"
    sbert_ef = SentenceTransformerEmbeddingFunction(model_name=model_name)
    # Check embedding model info
    print(f"🤖 Using model: {model_name}")
    print(f"📐 Embedding dimensions: {sbert_ef._model.get_embedding_dimension()}")

    # Create collection (what is the collection name?)
    # hnsw: Hierarchical Navigable Small World: default algorithm
    try:
        collection = client.create_collection(
            name="techcorp_policies",
            metadata={"hnsw:space": "ip"},
            embedding_function=sbert_ef,
        )
    except Exception:
        # Collection already exists
        collection = client.get_collection("techcorp_policies")

    ids = [chunk["id"] for chunk in chunks]
    documents = [chunk["content"] for chunk in chunks]
    metadatas = [{"title": chunk["title"], "category": chunk["category"], "source": chunk["source_doc"]} for chunk in chunks]

    # Add documents to collection (embeddings of ChromaDB will be generated automatically, default all-MiniLM-L6-v2)
    if collection.count() == 0:
        collection.add(
            ids=ids,
            documents=documents,
            metadatas=metadatas,
        )
        print(f"✅ Stored {len(chunks)} chunks in vector database")
    else:
        print(f"✅ Collection already contains {collection.count()} chunks")

    print(f"📈 Collection count: {collection.count()}")

    return collection


In [295]:
def process_user_query(query: str):
  print("\n🔍 SECTION 3: QUERY PROCESSING")
  print("-" * 50)
  cleaned_query = query.lower().strip()
  print(f"🧹 Preprocessed query: {cleaned_query}")

  model = SentenceTransformer('all-MiniLM-L6-v2')
  query_embedding = model.encode([cleaned_query])
  print(f"🔢 Query embedding shape: {query_embedding.shape}")
  print(f"📊 Embedding sample: {query_embedding[0][:5]}...")

  return model, query_embedding[0]

In [296]:
def search_vector_database(collection, query_embedding, top_k: int = 3):
    """
    Search vector database for relevant document chunks.

    This section demonstrates:
    - Vector similarity search
    - Result ranking and filtering
    - Similarity scoring
    - Top-k result selection
    """
    print("\n🔍 SECTION 4: VECTOR SEARCH")
    print("-" * 50)
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )
    print(f"🎯 Searching for top {top_k} results")
    print(f"📊 Found {len(results['ids'][0])} relevant chunks")
    print(json.dumps(results, indent=4))
    search_results = []
    for i, (doc_id, distance, content, metadata) in enumerate(zip(
        results['ids'][0],
        results['distances'][0],
        results['documents'][0],
        results['metadatas'][0] )):
      similarity = 1 - distance  # Convert distance to similarity
      search_results.append({
          'id': doc_id,
          'content': content,
          'metadata': metadata,
          'similarity': similarity
      })
      print(f"   Similarity: {similarity:.3f}")
      print(f"   Content: {content[:100]}...")


In [297]:
def run_complete_rag_pipeline(query: str):
  # Step 1: Load and chunk documents
  chunks = load_and_chunk_documents()
  # Step 2: Setup vector database
  collection = setup_vector_database(chunks)
  # Step 3: Preprocess user query
  model, query_embedding = process_user_query(query)
  # Step 4: Search vector database
  search_results = search_vector_database(collection, query_embedding)


In [298]:
test_queries = [
        "What's the reimbursement policy for home office equipment?",
        "Can I get money back for buying a desk?",
        "How much can I claim for my home office?",
        "What's the travel expense policy?",
        "How many vacation days do I get?"
    ]

for i, query in enumerate(test_queries, 1):
        print(f"\n{'='*60}")
        print(f"DEMO {i}: {query}")
        print(f"{'='*60}")
        try:
            run_complete_rag_pipeline(query)
        except Exception as e:
            print(f"❌ Error in demo {i}: {e}")



DEMO 1: What's the reimbursement policy for home office equipment?
📚 SECTION 1: DOCUMENT LOADING & CHUNKING
--------------------------------------------------
📄 Loaded 5 policy documents
Splited to 13 chunks

🗄️ SECTION 2: VECTOR DATABASE SETUP
--------------------------------------------------
🤖 Using model: all-MiniLM-L6-v2
📐 Embedding dimensions: 384
✅ Collection already contains 13 chunks
📈 Collection count: 13

🔍 SECTION 3: QUERY PROCESSING
--------------------------------------------------
🧹 Preprocessed query: what's the reimbursement policy for home office equipment?


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔢 Query embedding shape: (1, 384)
📊 Embedding sample: [-0.05989287  0.08903448  0.03420068 -0.04238869 -0.02239706]...

🔍 SECTION 4: VECTOR SEARCH
--------------------------------------------------
🎯 Searching for top 3 results
📊 Found 3 relevant chunks
{
    "ids": [
        [
            "policy_004_chunk_0",
            "policy_001_chunk_0",
            "policy_003_chunk_1"
        ]
    ],
    "embeddings": null,
    "documents": [
        [
            "Work-related equipment and supplies purchased for home office use are eligible for reimbursement. This covers laptops, monitors, keyboards, mice, and other computer peripherals. Submit expense reports",
            "Employees working from home may claim up to $500 per year for office equipment including desks, chairs, monitors, and computer accessories. Receipts must be submitted within 30 days of purchase. This",
            "reimbursement is $300 per item with manager approval required. All furniture must meet ergonomic standards

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔢 Query embedding shape: (1, 384)
📊 Embedding sample: [ 0.05946587  0.04194313 -0.01191727  0.04838425  0.07805505]...

🔍 SECTION 4: VECTOR SEARCH
--------------------------------------------------
🎯 Searching for top 3 results
📊 Found 3 relevant chunks
{
    "ids": [
        [
            "policy_002_chunk_0",
            "policy_002_chunk_2",
            "policy_002_chunk_1"
        ]
    ],
    "embeddings": null,
    "documents": [
        [
            "Business travel expenses are reimbursable when pre-approved by your manager. Meals are covered up to $50 per day, hotel stays up to $200 per night. All receipts must be submitted within 14 days of",
            "is mandatory for all business trips exceeding 7 days.",
            "All receipts must be submitted within 14 days of return. International travel requires additional approval from the department head. Travel insurance is mandatory for all business trips exceeding 7"
        ]
    ],
    "uris": null,
    "included": [
    

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔢 Query embedding shape: (1, 384)
📊 Embedding sample: [ 0.0491493   0.01118137  0.02954275  0.0692545  -0.0339082 ]...

🔍 SECTION 4: VECTOR SEARCH
--------------------------------------------------
🎯 Searching for top 3 results
📊 Found 3 relevant chunks
{
    "ids": [
        [
            "policy_005_chunk_0",
            "policy_002_chunk_2",
            "policy_002_chunk_0"
        ]
    ],
    "embeddings": null,
    "documents": [
        [
            "Full-time employees accrue 15 days of paid time off per year. Vacation requests must be submitted at least 2 weeks in advance. Unused PTO does not roll over to the next year. Emergency leave can be",
            "is mandatory for all business trips exceeding 7 days.",
            "Business travel expenses are reimbursable when pre-approved by your manager. Meals are covered up to $50 per day, hotel stays up to $200 per night. All receipts must be submitted within 14 days of"
        ]
    ],
    "uris": null,
    "included": [
    